In [1]:
import sys
!{sys.executable} -m pip install ultralytics pandas matplotlib opencv-python seaborn torch pyyaml

import os
import yaml
import time
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

print("✅ Libraries installed and ready.")

✅ Libraries installed and ready.


In [2]:
pip install easyocr

  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl (40.1 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\souradip\\Desktop\\MAJOR_PROJECT\\yolo_env\\Lib\\site-packages\\cv2\\cv2.pyd'
Check the permissions.



In [8]:
import os
import yaml

# 🟢 CONFIGURATION
DATASET_ROOT_DIR = "datasets2"  
YAML_FILENAME = "data.yaml"

def make_paths_absolute(dataset_root, yaml_name):
    yaml_path = os.path.abspath(os.path.join(dataset_root, yaml_name))
    
    if not os.path.exists(yaml_path):
        print(f"❌ Error: Could not find {yaml_path}")
        print(f"   Current working directory: {os.getcwd()}")
        print(f"   Contents of '{dataset_root}': {os.listdir(dataset_root) if os.path.exists(dataset_root) else 'Folder not found'}")
        return None

    print(f"✅ Found config: {yaml_path}")
    
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # Helper to find a folder locally
    def find_folder(target_names, search_path):
        for name in target_names:
            candidate = os.path.join(search_path, name)
            if os.path.exists(candidate):
                return candidate
            # Try looking in 'images' subdir just in case
            candidate_img = os.path.join(search_path, name, "images")
            if os.path.exists(candidate_img):
                return candidate_img
        return None

    # 1. Fix TRAIN path
    train_path = find_folder(["train", "training"], os.path.abspath(dataset_root))
    if train_path:
        data['train'] = train_path # Set absolute path
        print(f"   -> Fixed TRAIN path: {train_path}")
    else:
        print("❌ Critical Error: Could not find a 'train' folder in datasets2.")

    # 2. Fix VAL/VALID path
    val_path = find_folder(["valid", "val", "validation"], os.path.abspath(dataset_root))
    if val_path:
        data['val'] = val_path # Set absolute path
        print(f"   -> Fixed VAL path:   {val_path}")
    else:
        # Fallback: use train path as val if missing (prevents crash, technically works for debugging)
        print("⚠️  Warning: 'valid' folder not found. Using 'train' folder for validation to prevent crash.")
        data['val'] = train_path

    # 3. Fix TEST path (optional)
    test_path = find_folder(["test", "testing"], os.path.abspath(dataset_root))
    if test_path:
        data['test'] = test_path
        print(f"   -> Fixed TEST path:  {test_path}")

    # Save the corrected YAML with absolute paths
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
    
    print("✅ data.yaml updated with ABSOLUTE paths. Ready to train!")
    return yaml_path

# Run the fix
DATA_YAML_PATH = make_paths_absolute(DATASET_ROOT_DIR, YAML_FILENAME)

✅ Found config: c:\Users\souradip\Desktop\MAJOR_PROJECT\datasets2\data.yaml
   -> Fixed TRAIN path: c:\Users\souradip\Desktop\MAJOR_PROJECT\datasets2\train
⚠️  Warning: 'valid' folder not found. Using 'train' folder for validation to prevent crash.
   -> Fixed TEST path:  c:\Users\souradip\Desktop\MAJOR_PROJECT\datasets2\test
✅ data.yaml updated with ABSOLUTE paths. Ready to train!


In [ ]:

EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16
PROJECT_DIR = "runs/detect" 

print(f"\n🚀 Starting YOLOv8 Training...")
start_time = time.time()


model_v8 = YOLO('yolov8n.pt') 

# Train
model_v8.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_DIR,
    name="yolov8n_plate_detect",
    exist_ok=True,
    plots=True
)

print(f"✅ YOLOv8 Training finished in {(time.time() - start_time)/60:.2f} minutes.")


🚀 Starting YOLOv8 Training...
Ultralytics 8.3.228  Python-3.11.2 torch-2.9.1+cpu CPU (11th Gen Intel Core i5-11400H @ 2.70GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\souradip\Desktop\vehicle classifcation\datasets2\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_plate_detect, nbs=64, nms=False, opset=None, opt

In [ ]:
print(f"\n🚀 Starting YOLOv5 Training...")
start_time = time.time()

model_v5 = YOLO('yolov5nu.pt') 

model_v5.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_DIR,
    name="yolov5n_plate_detect",
    exist_ok=True,
    plots=True
)

print(f"✅ YOLOv5 Training finished in {(time.time() - start_time)/60:.2f} minutes.")


🚀 Starting YOLOv5 Training...
Ultralytics 8.3.228  Python-3.11.2 torch-2.9.1+cpu CPU (11th Gen Intel Core i5-11400H @ 2.70GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\souradip\Desktop\vehicle classifcation\datasets2\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov5nu.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov5n_plate_detect, nbs=64, nms=False, opset=None, op

In [10]:
import os
import pandas as pd
from ultralytics import YOLO

print("\n📊 Running Validation to generate Metrics...")

PATH_V8 = "runs/detect/yolov8n_plate_detect/weights/best.pt"
PATH_V5 = "runs/detect/yolov5n_plate_detect/weights/best.pt"
DATA_YAML = "datasets2/data.yaml"

def get_val_metrics(model_path, yaml_path):
    if not os.path.exists(model_path):
        print(f"⚠️ Weights not found: {model_path}")
        return None
    
    model = YOLO(model_path)

    metrics = model.val(data=yaml_path, verbose=False)
    
    return {
        "mAP50": metrics.box.map50,     
        "mAP50-95": metrics.box.map,    
        "Precision": metrics.box.mp,    
        "Recall": metrics.box.mr,       
        "Speed": metrics.speed['inference'], 
        "save_dir": metrics.save_dir   
    }

# 1. Get Metrics
stats_v8 = get_val_metrics(PATH_V8, DATA_YAML)
stats_v5 = get_val_metrics(PATH_V5, DATA_YAML)

# 2. Create Comparison Table
print("\n" + "="*85)
print(f"{' ':25} | {'YOLOv8':<25} | {'YOLOv5':<25}")
print(f"{'METRIC':<25} | {'yolov8n-plate':<25} | {'yolov5n-plate':<25}")
print("="*85)

if stats_v8 and stats_v5:
    print(f"{'mAP@50 (Accuracy)':<25} | {stats_v8['mAP50']:.4f}{' ':<19} | {stats_v5['mAP50']:.4f}")
    print(f"{'mAP@50-95 (Strict)':<25} | {stats_v8['mAP50-95']:.4f}{' ':<19} | {stats_v5['mAP50-95']:.4f}")
    print(f"{'Precision':<25} | {stats_v8['Precision']:.4f}{' ':<19} | {stats_v5['Precision']:.4f}")
    print(f"{'Recall':<25} | {stats_v8['Recall']:.4f}{' ':<19} | {stats_v5['Recall']:.4f}")
    print(f"{'Inference Speed':<25} | {stats_v8['Speed']:.1f} ms{' ':<17} | {stats_v5['Speed']:.1f} ms")
else:
    print("❌ Error: Could not find trained weights for one or both models.")
print("="*85)


📊 Running Validation to generate Metrics...
Ultralytics 8.4.14  Python-3.11.2 torch-2.7.1+cu118 CPU (11th Gen Intel Core i5-11400H @ 2.70GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 65.928.2 MB/s, size: 50.0 KB)
val: Scanning C:\Users\souradip\Desktop\MAJOR_PROJECT\datasets2\train\labels.cache... 336 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 336/336  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 533, len(boxes) = 707. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.5s/it 31.5s1.4ss
                   all        336        707      0.923      0.857      0.942      0.841
Speed: 1.1ms preprocess, 76.7ms inference, 0.0ms los

In [3]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import random
import os
import re
from ultralytics import YOLO


def preprocess_for_ocr(img):

    if img is None or img.size == 0:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Upscale x2
    gray = cv2.resize(
        gray,
        (gray.shape[1] * 2, gray.shape[0] * 2),
        interpolation=cv2.INTER_CUBIC
    )

    # Mild blur
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    return gray


def clean_plate_text(text):

    if not text:
        return ""
    text = text.upper()
    return re.sub(r"[^A-Z0-9]", "", text)


In [4]:

try:
    import easyocr
    import torch
    use_gpu = torch.cuda.is_available()
    ocr_reader = easyocr.Reader(['en'], gpu=use_gpu)
    print(f"✔ EasyOCR initialized (GPU={use_gpu})")
except Exception as e:
    print("⚠ EasyOCR not available:", e)
    ocr_reader = None


Using CPU. Note: This module is much faster with a GPU.


✔ EasyOCR initialized (GPU=False)


In [5]:

TEST_IMAGES_DIR = "dataset_test"
MODEL_WEIGHTS_PATH = "runs/detect/yolov8n_plate_detect/weights/best.pt"

# Validate folders
if not os.path.isdir(TEST_IMAGES_DIR):
    raise Exception(f"❌ TEST FOLDER MISSING: {TEST_IMAGES_DIR}")

if not os.path.exists(MODEL_WEIGHTS_PATH):
    raise Exception(f"❌ MODEL WEIGHTS NOT FOUND: {MODEL_WEIGHTS_PATH}")

# Load Model
model_yolo = YOLO(MODEL_WEIGHTS_PATH)
print("✔ YOLOv8 Model Loaded Successfully!")


✔ YOLOv8 Model Loaded Successfully!


In [9]:

image_files = [
    f for f in os.listdir(TEST_IMAGES_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

if not image_files:
    raise Exception("❌ No test images found!")

img_name = random.choice(image_files)
img_path = os.path.join(TEST_IMAGES_DIR, img_name)

print(f"🔍 Processing: {img_name}")

# Read original
img_bgr = cv2.imread(img_path)

# 2️⃣ YOLO detection
pred = model_yolo.predict(img_path, conf=0.25, save=False, verbose=False)
img_yolo_bgr = pred[0].plot()
img_yolo_rgb = cv2.cvtColor(img_yolo_bgr, cv2.COLOR_BGR2RGB)

# 3️⃣ Crop plate region
detected_text = ""
cropped_plate_rgb = None

if len(pred[0].boxes) > 0:
    box = pred[0].boxes[0]    # take highest confidence
    print(f"🚗 Plate detected (Conf={box.conf[0]:.2f})")

    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

    # Add padding
    pad = 5
    H, W, _ = img_bgr.shape
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(W, x2 + pad), min(H, y2 + pad)

    cropped_bgr = img_bgr[y1:y2, x1:x2]

    if cropped_bgr.size > 0:
        cropped_plate_rgb = cv2.cvtColor(cropped_bgr, cv2.COLOR_BGR2RGB)

        # 4️⃣ Soft Preprocessing
        ocr_ready = preprocess_for_ocr(cropped_bgr)

        # 5️⃣ OCR
        if ocr_reader:
            print("\n--- OCR Raw Results ---")
            ocr_results = ocr_reader.readtext(ocr_ready)

            raw_text = ""
            for (bbox, txt, prob) in ocr_results:
                print(f"> {txt} (Conf {prob:.2f})")
                if prob > 0.10:        
                    raw_text += txt + " "

            detected_text = clean_plate_text(raw_text)
            print(f"🧹 Cleaned Plate Text: {detected_text}")

else:
    print("ℹ No number plate detected.")


# 6️⃣ Final Visualization
plt.figure(figsize=(16, 8))

# YOLO Output
plt.subplot(1, 2, 1)
plt.imshow(img_yolo_rgb)
plt.title("YOLOv8 Number Plate Detection")
plt.axis("off")

# Plate Crop + OCR text
plt.subplot(1, 2, 2)
if cropped_plate_rgb is not None:
    t = "Extracted Plate"
    if detected_text:
        t += f"\nOCR: {detected_text}"
    plt.imshow(cropped_plate_rgb)
    plt.title(t, fontsize=14, fontweight="bold", color="green")
else:
    plt.imshow(img_yolo_rgb)
    plt.title("No Plate Found")

plt.axis("off")
plt.show()


Exception: ❌ No test images found!